**Raw Text**
```
  → [Pre-tokenization] 
  → [Normalization] 
  → [Subword Splitting via Vocab] 
  → Tokens 
  → [Convert to IDs] 
  → IDs 
  → [Embedding Layer] 
  → Dense Vectors 
  → [Transformer Model]
```

In [1]:
from transformers import pipeline

# 1. Load sentiment-analysis pipeline
classifier = pipeline("sentiment-analysis")

# 2. Try with simple sentences
print(classifier("I love learning NLP!"))
print(classifier("This Dashain is making me too busy."))

# 3. Batch input
texts = [
    "PyTorch feels intuitive to me.",
    "I am scared I might fail at becoming an NLP engineer.",
    "The food during Dashain is amazing!"
]
print(classifier(texts))


No model was supplied, defaulted to distilbert-base-uncased-finetuned-sst-2-english and revision af0f99b (https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
c:\Users\bisha\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

c:\Users\bisha\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bisha\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Xformers is not installed correctly. If you want to use memory_efficient_attention to accelerate training use the following command to install Xformers
pip install xformers.


[{'label': 'POSITIVE', 'score': 0.9995231628417969}]
[{'label': 'NEGATIVE', 'score': 0.9996249675750732}]
[{'label': 'POSITIVE', 'score': 0.998793363571167}, {'label': 'NEGATIVE', 'score': 0.9993988275527954}, {'label': 'POSITIVE', 'score': 0.9998855590820312}]


In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "Unbelievable! NLP is evolving fast."

# Inspect how BERT tokenizes
tokens = tokenizer.tokenize(text)
ids = tokenizer.convert_tokens_to_ids(tokens)

print("Tokens:", tokens)
print("IDs:", ids)

# Tokens: ['unbelievable', '!', 'nl', '##p', 'is', 'evolving', 'fast', '.']
# IDs: [23653, 999, 17953, 2361, 2003, 20607, 3435, 1012]


Tokens: ['unbelievable', '!', 'nl', '##p', 'is', 'evolving', 'fast', '.']
IDs: [23653, 999, 17953, 2361, 2003, 20607, 3435, 1012]


In [5]:
encoded = tokenizer("Hello world")
print(encoded)  # input_ids, attention_mask

# Show embeddings (needs model)
from transformers import AutoModel
import torch

model = AutoModel.from_pretrained("bert-base-uncased")
# Convert lists in encoded to tensors
encoded_tensors = {k: torch.tensor([v]) if isinstance(v, list) else v for k, v in encoded.items()}
with torch.no_grad():
    outputs = model(**encoded_tensors)
print(outputs.last_hidden_state.shape)  # [batch, seq_len, hidden_size]


{'input_ids': [101, 7592, 2088, 102], 'token_type_ids': [0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1]}


Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


torch.Size([1, 4, 768])


In [6]:
print("Vocab size:", tokenizer.vocab_size)
print("Does 'aspirin' exist in vocab?", "aspirin" in tokenizer.vocab)
print("How is 'acetylsalicylic' tokenized?:", tokenizer.tokenize("acetylsalicylic"))


Vocab size: 30522
Does 'aspirin' exist in vocab? False
How is 'acetylsalicylic' tokenized?: ['ace', '##ty', '##ls', '##alic', '##yl', '##ic']


In [7]:
encoded = tokenizer("I love NLP", padding="max_length", max_length=8)
print(encoded)
print("Tokens with specials:", tokenizer.convert_ids_to_tokens(encoded["input_ids"]))


{'input_ids': [101, 1045, 2293, 17953, 2361, 102, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 0, 0]}
Tokens with specials: ['[CLS]', 'i', 'love', 'nl', '##p', '[SEP]', '[PAD]', '[PAD]']


In [8]:
encoded = tokenizer(["short text", "this is a longer text"], padding=True, truncation=True)
print(encoded)


{'input_ids': [[101, 2460, 3793, 102, 0, 0, 0], [101, 2023, 2003, 1037, 2936, 3793, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1]]}
